In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

import pandas as pd
import numpy as np
import torch

In [2]:
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
# device = u.Devices().auto_set_device()#drop=['cuda:4'])

In [19]:
def list_tcga(tcga_dir:str):
    tcga_dir = Path(tcga_dir)

    # return TCGA projects with both metadata and gene counts
    metadata = [f.name.split('_metadata.csv')[0] for f in tcga_dir.glob('*_metadata.csv')]
    gene_counts = [f.name.split('_gene_counts.csv')[0] for f in tcga_dir.glob('*_gene_counts.csv')]
    return list(set(metadata) & set(gene_counts))

def preview_tcga(tcga_project:str, tcga_dir:str, return_df:bool=True, verbose:bool=True):
    # read df
    df = pd.read_csv(Path(tcga_dir) / f'{tcga_project}_metadata.csv')

    # drop 'Unnamed: 0' col if applicable
    if 'Unnamed: 0' in df.columns:
            df = df.drop(columns='Unnamed: 0')

    if verbose:
        # for each metadata column
        for i in df.columns:
            # get unique items per column
            unique = df[i].unique().tolist()

            # print items per column
            if len(unique) > 15:
                print(f'{i} ({len(unique)}): {unique[0:3]+['...']}\n') # shorten if > 15 unique
            else:
                print(f'{i} ({len(unique)}): {unique}\n')

    if return_df:
        return df

def preview_tcga_subtypes(tcga_project:str, tcga_dir:str, subtype_col:str='paper_Original.Subtype', return_df:bool=False):
    df = preview_tcga(tcga_project, tcga_dir, return_df=True, verbose=False)

    for i in df['tissue_type'].unique():
        print(i, df[df['tissue_type']==i][subtype_col].value_counts(dropna=False).to_dict())    
    print()

    for i in df['sample_type'].unique():
        print(i, df[df['sample_type']==i][subtype_col].value_counts(dropna=False).to_dict())

    if return_df:
        return df

In [12]:
projects = list_tcga(dataset_dir/'tcga')
projects

['TCGA-OV', 'TCGA-KIRC', 'TCGA-BRCA', 'TCGA-LGG', 'TCGA-GBM', 'TCGA-HNSC']

In [24]:
for project in projects:
    d = preview_tcga(project, dataset_dir/'tcga', return_df=True, verbose=False)
    print(f'{project}: {d.shape}')
    print(d['tissue_type'].value_counts().to_dict())
    print(d['sample_type'].value_counts().to_dict())
    if 'paper_Original.Subtype' in d.columns:
        print(d['paper_Original.Subtype'].value_counts().to_dict())
        # preview_tcga_subtypes(tcga_project=project, tcga_dir=dataset_dir/'tcga', subtype_col='paper_Original.Subtype')
    print('---')

TCGA-OV: (429, 67)
{'Tumor': 429}
{'Primary Tumor': 421, 'Recurrent Tumor': 8}
---
TCGA-KIRC: (614, 71)
{'Tumor': 542, 'Normal': 72}
{'Primary Tumor': 541, 'Solid Tissue Normal': 72, 'Additional - New Primary': 1}
---
TCGA-BRCA: (1231, 93)
{'Tumor': 1118, 'Normal': 113}
{'Primary Tumor': 1111, 'Solid Tissue Normal': 113, 'Metastatic': 7}
---
TCGA-LGG: (534, 111)
{'Tumor': 534}
{'Primary Tumor': 516, 'Recurrent Tumor': 18}
{'IDHmut-non-codel': 249, 'IDHmut-codel': 169, 'IDHwt': 95}
---
TCGA-GBM: (391, 106)
{'Tumor': 386, 'Normal': 5}
{'Primary Tumor': 372, 'Recurrent Tumor': 14, 'Solid Tissue Normal': 5}
{'Mesenchymal': 106, 'Classical': 89, 'Neural': 66, 'Proneural': 65, 'G-CIMP': 21}
---
TCGA-HNSC: (566, 85)
{'Tumor': 522, 'Normal': 44}
{'Primary Tumor': 520, 'Solid Tissue Normal': 44, 'Metastatic': 2}
---


In [20]:
d = preview_tcga(tcga_project='TCGA-BRCA', tcga_dir=dataset_dir/'tcga', verbose=False)
# brca (n=1231)
# drop = ['Normal', 'Primary Tumor', 'Metastatic']

# cols:
# paper_BRCA_Subtype_PAM50: ['LumA', 'Her2', 'LumB', 'Basal', nan, 'Normal']

preview_tcga_subtypes(tcga_project='TCGA-BRCA', tcga_dir=dataset_dir/'tcga', subtype_col='paper_BRCA_Subtype_PAM50')

Tumor {'LumA': 571, 'LumB': 209, 'Basal': 197, 'Her2': 82, 'Normal': 40, nan: 19}
Normal {nan: 113}

Primary Tumor {'LumA': 571, 'LumB': 209, 'Basal': 197, 'Her2': 82, 'Normal': 40, nan: 12}
Solid Tissue Normal {nan: 113}
Metastatic {nan: 7}


In [21]:
d = preview_tcga(tcga_project='TCGA-GBM', tcga_dir=dataset_dir/'tcga', verbose=False)
# gbm

# d['paper_Original.Subtype'].value_counts(dropna=False).to_dict()

preview_tcga_subtypes(tcga_project='TCGA-GBM', tcga_dir=dataset_dir/'tcga')

Tumor {'Mesenchymal': 106, 'Classical': 89, 'Neural': 66, 'Proneural': 65, nan: 39, 'G-CIMP': 21}
Normal {nan: 5}

Primary Tumor {'Mesenchymal': 106, 'Classical': 89, 'Neural': 66, 'Proneural': 65, nan: 25, 'G-CIMP': 21}
Recurrent Tumor {nan: 14}
Solid Tissue Normal {nan: 5}


In [25]:
d = preview_tcga(tcga_project='TCGA-LGG', tcga_dir=dataset_dir/'tcga', verbose=False)
# lgg

# d['paper_Histology'].value_counts()
# d['paper_Original.Subtype'].value_counts()

preview_tcga_subtypes(tcga_project='TCGA-LGG', tcga_dir=dataset_dir/'tcga')

Tumor {'IDHmut-non-codel': 249, 'IDHmut-codel': 169, 'IDHwt': 95, nan: 21}

Primary Tumor {'IDHmut-non-codel': 249, 'IDHmut-codel': 169, 'IDHwt': 95, nan: 3}
Recurrent Tumor {nan: 18}
